In [1]:
import geopandas as gpd
import pandas as pd
import re
from docx import Document
from rapidfuzz import fuzz, process

In [108]:
data_path = '../../../data/shapes'
shp_path = '../../../data/shapes/farmers_polygons.shp'
farmers_list = '../../../data/shapes/farmers_list.docx'

In [60]:
gdf = gpd.read_file(shp_path)

In [61]:
gdf.columns

Index(['id', 'name', 'melia_only', 'bounds_ok', 'geometry'], dtype='str')

In [62]:
gdf = gdf[['name', 'melia_only', 'bounds_ok', 'geometry']]

In [63]:
gdf[gdf['name'].isna()]

,name,melia_only,bounds_ok,geometry
72,NaN,False,True,"POLYGON ((372927.037 9823871.628, 372996.093 9..."
118,NaN,False,False,"POLYGON ((374169.813 9826072.278, 374265.336 9..."


In [99]:
doc = Document(farmers_list)

In [100]:
rows = []
for table in doc.tables:
    for row in table.rows[1:]:  # skip header
        cells = row.cells
        rows.append({
            "name": cells[1].text,
            "trees_given": cells[2].text,
            "trees_alive": cells[3].text
        })

doc_df = pd.DataFrame(rows)

In [101]:
doc_df.head()

,name,trees_given,trees_alive
0,Names,,
1,Rose Martha Mulaa,350,350
2,John Kithuku Mulu,150,104
3,Stella Kavindu Isaac,200,194
4,Priscah Martha Mutia,150,156


In [102]:
doc_df = doc_df.drop(index=0)

#### Name normalization

In [68]:
# Define unwanted substrings / tokens
UNWANTED_TERMS = [
    "awg",
    "farmer",
    "old farmer",
    "new",
    "ond",
    '-'
]

In [69]:
def normalize_name(name):
    if pd.isnull(name):
        return None
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # remove punctuation
    for term in sorted(UNWANTED_TERMS, key=len, reverse=True):
        pattern = r'\b' + re.escape(term) + r'\b'
        name = re.sub(pattern, '', name)
    
    # Remove standalone 4-digit years
    name = re.sub(r'\b\d{4}\b', '', name)
    name = re.sub(r'\s+', ' ', name)     # collapse spaces
    return name.strip()

In [70]:
len(gdf)

153

In [ ]:
# Remove null names
gdf = gdf[gdf["name"].notnull()]

# Split grouped names
gdf["name_split"] = gdf["name"].str.split(",")

# Explode into individual rows
gdf = gdf.explode("name_split")

# Normalize
gdf["name_clean"] = gdf["name_split"].apply(normalize_name)

# Remove empty
gdf = gdf[gdf["name_clean"].notnull()]

In [72]:
len(gdf)

157

In [84]:
gdf.columns

Index(['name', 'melia_only', 'bounds_ok', 'geometry', 'name_split',
       'name_clean'],
      dtype='str')

#### Clean docx names

In [103]:
len(doc_df)

174

In [104]:
doc_df["name_clean"] = doc_df["name"].apply(normalize_name)

In [113]:
doc_df[['name', 'name_clean']].iloc[:20]

,name,name_clean
1,Rose Martha Mulaa,rose martha mulaa
2,John Kithuku Mulu,john kithuku mulu
3,Stella Kavindu Isaac,stella kavindu isaac
4,Priscah Martha Mutia,priscah martha mutia
5,Daina Nditi Mutisya Ngambi,daina nditi mutisya ngambi
6,Justus Mutinda Mutia,justus mutinda mutia
7,Jeremiah Ngambi Mutisya,jeremiah ngambi mutisya
8,Maurice Mulaa Manzolo,maurice mulaa manzolo
9,Pius Mutia Mathuku,pius mutia mathuku
10,Lenah Koki Mutinda,lenah koki mutinda


In [106]:
len(doc_df)

174

#### Fuzzy matching

In [114]:
threshold = 70

shp_names = gdf["name_clean"].unique().tolist()
used_shp_names = set()

matches = []
unmatched_docx = []

for _, row in doc_df.iterrows():
    doc_name = row["name_clean"]

    if pd.isnull(doc_name):
        continue

    result = process.extractOne(
        doc_name,
        shp_names,
        scorer=fuzz.token_sort_ratio
    )

    if result:
        matched_name, score, _ = result

        if score >= threshold:
            matches.append({
                "docx_name": row["name"],
                "docx_clean": doc_name,
                "matched_shapefile_name": matched_name,
                "similarity_score": score
            })
            used_shp_names.add(matched_name)
        else:
            unmatched_docx.append(row["name"])
    else:
        unmatched_docx.append(row["name"])
        
matches_df = pd.DataFrame(matches)
unmatched_df = pd.DataFrame(unmatched_docx, columns=["name_not_in_shapefile"])

#### Output files

In [119]:
shapefile_only = list(set(shp_names) - used_shp_names)
len(shapefile_only)

shapefile_only_df = pd.DataFrame(
    shapefile_only,
    columns=["name_in_shapefile_not_in_docx"]
)
shapefile_only_df.head()
shapefile_only_df.to_excel("shapefile_not_in_docx.xlsx", index=False)

In [122]:
# A. Matched for manual review
matches_df.to_excel(f"{data_path}/matched_names_for_review.xlsx", index=False)

# B. Names in DOCX not in shapefile
unmatched_df.to_excel(f"{data_path}/docx_not_in_shapefile.xlsx", index=False)

# C. Normalized shapefile
gdf[['name_clean']].to_excel(f"{data_path}/shapefile_names.xlsx")